In [1]:
#Funciones auxiliares


import numpy as np

def house(x):
  '''
  u, rho = house(x)
  Calcula u y rho tal que Q = I - rho u u^T
  cumple Qx = |x|_2 e^1
  '''
  n = len(x)
  rho = 0
  u = x.copy()
  u[0] = 1.

  if n == 1:
    sigma = 0
  else:
    sigma = np.sum(x[1:]**2)

  if sigma>0 or x[0]<0:
    mu = np.sqrt(x[0]**2 + sigma)
    if x[0]<=0:
      gamma = x[0] - mu
    else:
      gamma = -sigma/(x[0] + mu)

    rho = 2*gamma**2/(gamma**2 + sigma)
    u = u/gamma
    u[0] = 1

  return u, rho

def givens(x1,x2):
    '''
    c, s = givens(x1, x2)
    Calcula el coseno y seno para la rotación de Givens
    que hace (x1,x2) -> (y,0).
    '''
    c = 1.
    s = 0.
    ax1 = abs(x1)
    ax2 = abs(x2)
    if ax1 + ax2 > 0:
        if ax2 > ax1:
            tau = -x1/x2
            s = -np.sign(x2)/np.sqrt(1 + tau**2)
            c = tau*s
        else:
            tau = -x2/x1
            c = np.sign(x1)/np.sqrt(1 + tau**2)
            s = tau*c
    return c, s


def fhess(A, p):
    m, n = A.shape
    if m != n:
        print("La matriz no es cuadrada")
        return None
    Q = np.eye(m)
    H = A.copy()

    if p == 0: #Realiza ref Householder
        for j in range(n-2):
            #I = j+1: , J =j:
            u, rho = house(H[j+1:, j])
            w = rho * u
            H[j+1:, j:] = H[j+1:, j:] - np.outer(w, u.T @ H[j+1:, j:])
            H[:, j+1:] = H[:, j+1:] - H[:, j+1:]@ np.outer(w, u.T)
            Q[:, j+1:] = Q[:, j+1:] - Q[:, j+1:]@np.outer(w, u.T)

    elif p == 1: # Realiza rot Givens
        for j in range(n - 2):
            for i in range(j + 2, n):
                c, s = givens(H[j + 1, j], H[i, j])
                rot = np.array([[c, -s], [s, c]])
                H[[j + 1, i], j:] = rot @ H[[j + 1, i], j:]
                H[:, [j + 1, i]] = H[:, [j + 1, i]] @ rot.T
                Q[:, [j + 1, i]] = Q[:, [j + 1, i]] @ rot.T
    else:
        print("Elegir un p que sea 0 o 1")
        return None

    return Q, H

In [3]:
def autqr(A, err= 1e-10, M = 500):
    n = A.shape[0]
    Q, H = fhess(A, p=1)
    G_rot = np.zeros((n, 2)) # Aqui vamos a ir guardando c,s para luego generar G.T y multiplicar R por derecha

    for k in range(M):
        # item a)
        for j in range(n-1):
            c, s = givens(H[j, j], H[j+1, j])
            G_rot[j, :] = np.array([c, s]) #Aqui guardamos c,s en la matriz G_rot (no podemos generarlos despues
                                           # en el item b) pues estariamos calculando c, s sobre la matriz actualizada H)
            G_cs = np.array([[c, -s], [s, c]]) # Es la rotacion que aplicamos por izquierda de H
            H[[j, j+1], j:] = G_cs @ H[[j, j+1], j:]

        # item b)
        for l in range(n-1):
            c, s = G_rot[l, :] # tomamos c,s de G_rot
            G_cs = np.array([[c, -s], [s, c]]) # Es la rotacion que aplicamos por izquierda
            H[: , [l, l+1]] = H[: , [l, l+1]] @ G_cs.T
            Q[: , [l, l+1]] = Q[: , [l, l+1]] @ G_cs.T

        if np.linalg.norm(H - np.diag(np.diag(H)), 'fro') < err:
            print ("llegamos a la tolerancia")
            break

    return Q, H

# TEST
A = np.random.random((3, 3))
Q, H = autqr(A)

print("---------------------------------------------------------------------------------")
print(f"Autovalores de A usando autqr: diag(H) = {np.diag(H)}")
print("---------------------------------------------------------------------------------")
print(f"Autovalores de A usando linalg aut(A) ={np.linalg.eigvals(A)}")
print("---------------------------------------------------------------------------------")
print(f"||A-Q @ H @ Q.T||_fro = {np.linalg.norm(Q @ H @ Q.T - A)}")
print("---------------------------------------------------------------------------------")

---------------------------------------------------------------------------------
Autovalores de A usando autqr: diag(H) = [ 1.5757012  -0.35964526 -0.09880069]
---------------------------------------------------------------------------------
Autovalores de A usando linalg aut(A) =[ 1.5757012  -0.35964526 -0.09880069]
---------------------------------------------------------------------------------
||A-Q @ H @ Q.T||_fro = 2.9115585591251392e-15
---------------------------------------------------------------------------------
